# Bu notebook projede kullanılan dataların hazırlık süreçlerini kapsamaktadır

## Market - Ürün dataları

Market dataları için Ömer Çolakoğlunun https://www.kaggle.com/datasets/omercolakoglu/1m-rows-turkish-market-sales-dataset? datasetinden yararlandım özünde sipariş verileri olsada müşteri adres gibi kolonları excel üzerinde sildim

In [2]:
import polars as pl

df = pl.read_excel("kaggleSalesData.xlsx")

df_cleaned = (
    df
    
    .with_columns(
        pl.col("ITEMNAME")
        
        .str.replace_all(r"\*\d+\*", " ")
        
        .str.replace_all(r"(?i)\d+([.,]\d+)?\s*(KG|GR|LT|ML|G)\b", " ")
        
        .str.replace_all(r"[^\w\s]", " ")
    
        .str.to_lowercase()
        
        .str.replace_all(r"\s+", " ")
        .str.strip_chars()
        .alias("CLEANED_NAME")
    )
    
    
    .with_columns(
        pl.col("BRAND").str.to_lowercase().str.strip_chars().alias("BRAND_LOWER")
    )
)


df_cleaned = df_cleaned.with_columns(
    pl.struct(["CLEANED_NAME", "BRAND_LOWER"])
    .map_elements(
        lambda x: x["CLEANED_NAME"].replace(x["BRAND_LOWER"], "").replace("  ", " ").strip() 
        if x["BRAND_LOWER"] and isinstance(x["BRAND_LOWER"], str) 
        else x["CLEANED_NAME"],
        return_dtype=pl.Utf8
    )
    .alias("NO_BRAND_NAME")
)



df_final = df_cleaned.with_columns(
    (
        pl.col("CATEGORY3").fill_null("") + " " +
        pl.col("CATEGORY4").fill_null("") + " " + 
        pl.col("BRAND_LOWER").fill_null("") + " " + 
        pl.col("NO_BRAND_NAME")
    )
    .str.to_lowercase()
    
    
    .str.replace_all(r"\b\d+\b", " ")
    
    
    .str.replace_all(r"\b[a-z]\b", " ")
    
    .str.replace_all(r"\s+", " ")
    .str.strip_chars()
    .alias("TFIDF_TEXT")
)

print(df_final.select(["ITEMNAME", "TFIDF_TEXT"]).head(10))



shape: (10, 2)
┌─────────────────────────────────┬─────────────────────────────────┐
│ ITEMNAME                        ┆ TFIDF_TEXT                      │
│ ---                             ┆ ---                             │
│ str                             ┆ str                             │
╞═════════════════════════════════╪═════════════════════════════════╡
│ ASK GUNESE BENZER               ┆ kitap kitaplar kitaplar ask gu… │
│ DETAN ELEK.CIHAZ+YEDEK *24*     ┆ hasere olduruculer makine liki… │
│ ULKER 898-1 BOLERO MUZ KRE.GOF… ┆ gofret muzlu ulker bolero muz … │
│ S VERNEL 1 LT                   ┆ camasir yumusaticilar klasik v… │
│ BIZIM MUT DERMASON FASULYE 2.5… ┆ bakliyat fasulyeler bizim mut … │
│ ALLAH SAGLIK VERSIN             ┆ kitap kitaplar kitaplar allah … │
│ MILKA 83 GR BUBBLY SUT.BADEM. … ┆ cikolata tablet cikolata milka… │
│ IPANA D.MAC.100 ML PRO-EXP.PRO… ┆ dis macunlari macunlar ipana m… │
│ BIZIM MUT BULGUR PILAVLIK 1 KG… ┆ bakliyat bulgurlar bizim mut b… │
│ MAR

yukarıdaki süreçte veriler temizlenip tf-idf formatına uygun haline getirilmiştir artık ürün verilerimiz modelimize verilmeye hazırdır

In [3]:
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer

corpus = df_final["TFIDF_TEXT"].to_list()
vectorizer = TfidfVectorizer(ngram_range=(1, 2))
tfidf_matrix = vectorizer.fit_transform(corpus)


joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')
joblib.dump(tfidf_matrix, 'tfidf_matrix.pkl')


df_final.write_parquet('cleaned_dataframe.parquet')